In [4]:
from typing import TypedDict , Literal
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END,START
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field


In [5]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [6]:
class CategoryOutput(BaseModel):
    category:Literal["billing", "tech_support", "general_faq"]=Field(description="the category of the customer query")
parser=PydanticOutputParser(pydantic_object=CategoryOutput)


In [7]:
class Supportstate(TypedDict):
    query:str
    category:str
    response:str

In [8]:
def classifier_node(state:Supportstate):
   prompt = PromptTemplate(
        template="""
    Classify the customer query into exactly one category:
    - billing
    - tech_support
    - general_faq

    Customer query: {query}

    {format_instructions}
    """,
        input_variables=["query"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
   chain=prompt|llm|parser
   result=chain.invoke({"query":state["query"]})
   return {
      "category":result.category
   }



In [9]:
def billing_node(state: Supportstate):
    prompt = (
        "You are the billing specialist. Briefly help with payment "
        f"or refund issues: {state['query']}"
    )
    result = llm.invoke(prompt).content
    return {
        "response": f"[BILLING TEAM]: {result}"
    }


In [10]:
def techsupport_node(state: Supportstate):
    prompt = (
        "You are the specialist in tech support, briefly explain this query. "
        f"Troubleshooting example: {state['query']}"
    )
    result = llm.invoke(prompt).content
    return {
        "response": f"[TECH SUPPORT]: {result}"
    }


In [11]:
def FAQ_node(state:Supportstate):
    prompt = (
        "You are a customer service agent. Answer simply: "
        f"{state['query']}"
    )
    result = llm.invoke(prompt)
    return {"response": f"[GENERAL FAQ]: {result.content}"}

In [16]:
def router_query(state: Supportstate) -> Literal["billing", "tech_support", "FAQ"]:
    category = state["category"]
    if category == "billing":
        return "billing"
    elif category == "tech_support":
        return "tech_support"
    else:
        return "FAQ"


In [17]:
workflow=StateGraph(Supportstate)

In [18]:
workflow.add_node("classifier", classifier_node)
workflow.add_node("billing", billing_node)
workflow.add_node("tech_support", techsupport_node)
workflow.add_node("FAQ", FAQ_node)

workflow.add_edge(START, "classifier")

workflow.add_conditional_edges(
    "classifier",
    router_query,
    {
        "billing": "billing",
        "tech_support": "tech_support",
        "FAQ": "FAQ"
    }
)

workflow.add_edge("billing", END)
workflow.add_edge("tech_support", END)
workflow.add_edge("FAQ", END)
app=workflow.compile()

In [ ]:
input_query = [
    "My money was deducted twice from my credit card, I want a refund.",
    "The mobile app crashes every time I click on the login button.",
    "What are your working hours and return policy?"
]

for q in input_query:
    output = app.invoke({"query": q})
    print(f"Query: {q}")
    print(f"{output['response']}")
    print("-" * 60)


In [20]:
print(output)

{'query': 'What are your working hours and return policy?', 'category': 'general_faq', 'response': '[GENERAL FAQ]: **Working Hours**  \n- Monday\u202f–\u202fFriday: 9\u202fam\u202f–\u202f5\u202fpm (local time)  \n- Closed on weekends and public holidays  \n\n**Return Policy**  \n- Returns accepted within\u202f30\u202fdays of purchase  \n- Items must be unused, with original packaging and a receipt  \n- Refunds are issued to the original payment method after inspection  \n- For defective or incorrect items, we’ll cover return shipping; otherwise, you’re responsible for return postage.'}
